# B：缓存标题与复用普通 CPU 画布
同一次原 B 加载，先验证整幅 720p 像素一致，再比较原排版和缓存排版。算法/bit/HWH/官方 DMA、GPIO、HDMI、VDMA 不改。每个代码框只运行一次，等 In [数字] 且无错误再继续。

红色报错或一分钟无进展时保存输出并停止；不重复运行、不释放缓冲、不关闭失败内核、不切换 Overlay。原分段诊断已完成，本次是新排版的两组对比，尚无实际提速结论。

## 1. 准备函数
In [数字] 且无错误表示完成，无打印正常。

In [ ]:
from b_layout_board_test import prepare_layout, load_layout, start_layout, check_actual_pixels, compare_layout, finish_layout


## 2. 检查文件和普通画布
应出现 B_LAYOUT_PRECHECK_PASS。只读取元数据并验证 CPU 像素，不加载 FPGA，不分配 DMA/视频帧。缓存初始化耗时单独记录。

In [ ]:
if 'layout_context' in globals():
    raise RuntimeError('Already prepared; do not rerun')
layout_retained = []
layout_context = prepare_layout('.', layout_retained)


## 3. 加载一次
应出现 B_LAYOUT_LOAD_PASS；原 B 一次加载，尚未传输。

In [ ]:
load_layout(layout_context)


## 4. 短帧和 HDMI 准备
应出现 B_LAYOUT_HDMI_READY。显示器应显示蓝、绿、红和白色定位线，确认正常再继续。

In [ ]:
start_layout(layout_context)


## 5. 用真实 FPGA 输出核对整幅像素
应出现 B_LAYOUT_PIXELS_PASS。新旧排版在3个标记位置整幅720p逐像素一致，再进入计时。屏幕显示左原图、右绿色候选叠加。

In [ ]:
check_actual_pixels(layout_context)


## 6. 两组对比计时
先原排版，再缓存排版，每组3帧预热和30帧计时。每5帧打印进度，最终 B_LAYOUT_COMPARE_COMPLETE。两组保留相同逐字核验、哈希、原生显示成本；两组结束的整幅图像须一致。

In [ ]:
compare_layout(layout_context)
for group in layout_context['report']['layout']['groups']:
    summary = group['summary']
    print('{}: checked loop {:.6f} FPS'.format(group['name'], summary['loop_fps']))
    for name, stage in summary['stages'].items():
        print('  {}: {:.6f} ms, {:.3f}%'.format(name, stage['mean_ms'], stage['share_of_sample_time_percent']))
print('Raw timings saved; independent verification pending.')


## 7. 关闭并打包
应出现 B_LAYOUT_RESULTS_READY 并显示最后一帧预览。显示器无信号属于预期。Ctrl+S；File → Download as → Notebook 下载执行记录，回到列表下载 b_layout_bundle.zip。发布NOT_RUN和原JSON视觉PENDING保留，待独立核验；显示配置60Hz不等于系统60FPS。

In [ ]:
finish_layout(layout_context)
from IPython.display import Image, display
display(Image(filename=str(layout_context['directory'] / 'b_layout_cached_final.png')))
